# 06 — Feature Engineering

This notebook creates the features used to forecast Bitcoin and Ethereum returns.

The main feature groups are:

- **Market features** — price, returns, volume, volatility, and technical indicators.
- **Google Trends** — search-interest variables.
- **Fear & Greed Index** — market sentiment variables.

Features are created for **1-day, 7-day, and 14-day forecasting horizons**.

To avoid data leakage, all predictors use only information available at or before time \(t\). Future values are used only to construct the forecasting target.

# Feature Selection and Feature Engineering

This notebook uses the stationarity, correlation, and Granger causality results from Notebook 05 to guide feature engineering for **1-, 7-, and 14-day forecasts**.

All predictors use only past or current information. For 7- and 14-day horizons, the target is the cumulative future log return. Overlapping multi-day targets may be autocorrelated, but this does not represent data leakage.

## Setup

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, str(Path.cwd().parent))

from src import config
from src.analysis import feature_engineering as fe

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)


### Locating the project root

Uses the same `find_project_root` pattern as the causality analysis notebook, so this notebook can be run from any working directory without a hard-coded absolute path.

In [2]:
def find_project_root(start_path: Path, relative_dataset_path: Path) -> Path:
    """Search the current directory and its parents for the required dataset."""
    start_path = start_path.resolve()
    for candidate in [start_path, *start_path.parents]:
        if (candidate / relative_dataset_path).exists():
            return candidate
    raise FileNotFoundError(
        f"Could not find '{relative_dataset_path}' from '{start_path}' or any parent directory."
    )


DATASET_RELATIVE_PATH = Path("data/processed/merged_dataset.csv")
PROJECT_ROOT = find_project_root(Path.cwd(), DATASET_RELATIVE_PATH)
DATA_PATH = PROJECT_ROOT / DATASET_RELATIVE_PATH

CAUSALITY_TABLES_DIR = PROJECT_ROOT / "results" / "causality_analysis" / "tables"
FEATURE_RESULTS_DIR = PROJECT_ROOT / "results" / "feature_engineering"
FEATURE_TABLES_DIR = FEATURE_RESULTS_DIR / "tables"
FEATURES_DIR = PROJECT_ROOT / "data" / "processed" / "features"

FEATURE_TABLES_DIR.mkdir(parents=True, exist_ok=True)
FEATURES_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root         : {PROJECT_ROOT}")
print(f"Dataset path          : {DATA_PATH}")
print(f"Causality tables dir  : {CAUSALITY_TABLES_DIR}")
print(f"Feature tables dir    : {FEATURE_TABLES_DIR}")
print(f"Features output dir   : {FEATURES_DIR}")


Project root         : /Users/mas364/Desktop/AI_Crypto_Forecasting
Dataset path          : /Users/mas364/Desktop/AI_Crypto_Forecasting/data/processed/merged_dataset.csv
Causality tables dir  : /Users/mas364/Desktop/AI_Crypto_Forecasting/results/causality_analysis/tables
Feature tables dir    : /Users/mas364/Desktop/AI_Crypto_Forecasting/results/feature_engineering/tables
Features output dir   : /Users/mas364/Desktop/AI_Crypto_Forecasting/data/processed/features


## Loading and validating the dataset

The same merged dataset used throughout the dissertation (`data/processed/merged_dataset.csv`) is loaded and validated before any feature engineering begins: the date column must be datetime, observations must be in chronological order, and any missing calendar days or missing values are reported rather than silently handled.

In [3]:
df = pd.read_csv(DATA_PATH, parse_dates=["Date"])
df = df.sort_values("Date").reset_index(drop=True)

validation_report = fe.validate_input_data(df, config.FINAL_COLUMNS)

print("Dataset validation")
print("-------------------")
print(f"Rows                     : {validation_report['n_rows']}")
print(f"Columns                  : {validation_report['n_columns']}")
print(f"Date column is datetime  : {validation_report['is_datetime']}")
print(f"Chronological order      : {validation_report['is_chronological']}")
print(f"Duplicate dates          : {validation_report['duplicate_dates']}")
print(f"Date range               : {validation_report['date_min'].date()} to {validation_report['date_max'].date()}")
print(f"Missing calendar dates   : {validation_report['missing_calendar_dates']} -> {validation_report['missing_calendar_dates_list']}")
print(f"Missing values by column : {validation_report['missing_values_by_column']}")


Dataset validation
-------------------
Rows                     : 1799
Columns                  : 12
Date column is datetime  : True
Chronological order      : True
Duplicate dates          : 0
Date range               : 2021-06-01 to 2026-05-06
Missing calendar dates   : 2 -> [Timestamp('2022-05-23 00:00:00'), Timestamp('2024-10-26 00:00:00')]
Missing values by column : {'Bitcoin_Volume': 9, 'Ethereum_Volume': 9}


#### Interpretation

The two missing calendar dates (23 May 2022 and 26 October 2024) and the nine missing Bitcoin/Ethereum volume observations were already identified and documented during the data preparation stage; they are confirmed here, not newly discovered. The nine missing volume values correspond to the known corrupt Bitcoin/Ethereum volume window (1–9 April 2026), which was converted to NaN upstream during preprocessing rather than left as physically impossible figures.

This notebook linearly interpolates that nine-day volume gap before computing any volume-derived feature (lags, percentage change, log, rolling statistics) — a deliberate change from an earlier version of this pipeline, which carried the NaN through untouched and relied on dropping the affected rows later. That approach is sound for training, where dropping a handful of rows out of 1,700+ costs nothing, but breaks down for live single-window inference on newly uploaded data, where the affected window may be the only one available and there is no other row to fall back on. A short, isolated nine-day gap in an otherwise continuous daily series is a reasonable case for linear interpolation: volume moves gradually day to day, and the interpolated span is small relative to the ~30-day rolling window it would otherwise contaminate. The detection and masking decision itself (implausible volume → NaN) is unchanged and still lives upstream in preprocessing; only how feature engineering treats that masked gap has changed.

Structural missingness (the lead-in rows every lag/rolling feature needs, and the target's final undefined row) is unaffected by this change and is still removed explicitly once every feature and target has been aligned (see the missing-value summary near the end of this notebook).

### Removing market capitalisation, if present

Market capitalisation is not part of `FINAL_COLUMNS` in this project's configuration and was already excluded from the correlation and Granger causality analysis (Section 4.7) on the grounds of its near-identity with price. This check confirms it is not present here either, rather than assuming so.

In [4]:
market_cap_columns = [c for c in df.columns if "Market_Cap" in c]

if market_cap_columns:
    print(f"Removing market capitalisation columns: {market_cap_columns}")
    df = df.drop(columns=market_cap_columns)
else:
    print("No market capitalisation columns present in the feature-engineering input dataset.")


Removing market capitalisation columns: ['Bitcoin_Market_Cap', 'Ethereum_Market_Cap']


## Feature Selection Decision Table

The decision table below is constructed from the statistical outputs saved by the previous analysis notebook. The evidence is not recomputed or manually copied here.

The analyses have different roles:

- **Stationarity analysis** determines the transformation required before modelling, such as log returns or first differences.
- **Correlation analysis** measures contemporaneous linear and monotonic relationships.
- **Granger causality analysis** tests whether lagged values of one stationary series improve prediction of another stationary series.
- **Feature selection** uses these results to decide whether variables belong in the Market Only specification or should instead be evaluated experimentally.

The following rules are applied:

- Variables without significant correlation or Granger evidence are not treated as confirmed predictors.
- Google Trends is retained as an experimental signal because its evidence was comparatively stronger than that of Fear & Greed, although still non-significant.
- Fear & Greed is excluded from the Market Only specification but retained in its own feature configuration.
- Trading volume is retained as an internal market variable after appropriate stationary transformation; this is a modelling decision rather than evidence of Granger causality.
- Cross-asset return features are considered only after BTC → ETH and ETH → BTC Granger causality have been tested.

(The table below retains its original internal column names, defined in `src/analysis/feature_engineering.py` and unchanged here -- they correspond respectively to the Market Only, Market + Google Trends, and Market + Fear & Greed Index configurations described above.)

In [5]:
feature_selection_decisions = fe.build_feature_selection_decisions(CAUSALITY_TABLES_DIR)

decisions_path = FEATURE_TABLES_DIR / "feature_selection_decisions.csv"
feature_selection_decisions.to_csv(decisions_path, index=False)

print(f"Saved: {decisions_path}\n")
display(feature_selection_decisions[[
    "variable", "tested_against", "pearson_r", "pearson_significant_fdr",
    "spearman_r", "spearman_significant_fdr", "granger_p", "granger_significant_fdr",
    "selection_category", "keep_in_primary_set", "keep_in_candidate_set", "keep_in_ablation_set",
]])


Saved: /Users/mas364/Desktop/AI_Crypto_Forecasting/results/feature_engineering/tables/feature_selection_decisions.csv



,variable,tested_against,pearson_r,pearson_significant_fdr,spearman_r,spearman_significant_fdr,granger_p,granger_significant_fdr,selection_category,keep_in_primary_set,keep_in_candidate_set,keep_in_ablation_set
0,Bitcoin_Search_Interest,Bitcoin_Price,-0.056188,True,0.006228,False,0.135272,False,Candidate (weak/borderline evidence),False,True,False
1,Ethereum_Search_Interest,Ethereum_Price,-0.001728,False,0.088556,True,0.108815,False,Candidate (weak/borderline evidence),False,True,False
2,Fear_Greed_Index,Bitcoin_Price,-0.004990,False,-0.021764,False,0.919311,False,Excluded (no evidence),False,False,True
3,Fear_Greed_Index,Ethereum_Price,0.007746,False,0.003342,False,0.716223,False,Excluded (no evidence),False,False,True
4,Bitcoin_Volume,Bitcoin_Price,0.042667,False,0.008488,False,NaN,None,Not tested for causality (internal market vari...,True,False,False
5,Ethereum_Volume,Ethereum_Price,-0.035612,False,-0.016198,False,NaN,None,Not tested for causality (internal market vari...,True,False,False
6,Bitcoin_Price / Ethereum_Price (cross-asset),N/A,0.837861,True,NaN,None,NaN,None,Not tested (cross-asset Granger causality not ...,False,False,False


#### Interpretation

None of the external variables reached the predefined FDR-corrected significance threshold in the correlation or Granger causality analyses. The results therefore do not support treating Google Trends or the Fear & Greed Index as established linear predictors of cryptocurrency returns.

Google Trends nevertheless demonstrated comparatively stronger statistical evidence than the Fear & Greed Index, particularly in the Ethereum analysis, although the result remained non-significant. It is therefore retained in the Market + Google Trends configuration so that the modelling stage can test whether nonlinear algorithms identify predictive information not captured by the earlier linear tests.

The Fear & Greed Index did not demonstrate statistically significant correlation or Granger-causal relationships with the return targets. It is excluded from the Market Only and Market + Google Trends configurations but retained in its own Market + Fear & Greed Index configuration. This allows its incremental forecasting value to be tested without implying that the statistical analysis already supports it.

Trading volume was not evaluated as an external Granger-causal predictor in the previous analysis. Its inclusion in the Market Only configuration is instead based on its role as an internal market-activity variable and on the stationary transformation established earlier. This distinction prevents stationarity from being incorrectly described as evidence of predictive importance.

## Feature Engineering Principles

Before building any feature, the following rules are applied throughout this notebook and checked explicitly afterwards:

1. **No leakage.** Every lagged and rolling feature uses `.shift()` in the leakage-safe direction, and rolling windows are computed on an already-shifted series so that the statistic available at row *t* never includes row *t* itself.
2. **No shuffling.** The dataset stays in chronological order throughout.
3. **Parsimony.** Only a small, justified set of lags and windows is used — not an arbitrary grid.
4. **Evidence-selected lags are distinguished from conventional lags.** Lag 2 is used because it is the lag selected by the VAR/BIC analysis in Section 4.7; lags 1 and 7 are included as standard one-step and weekly conventions, with no statistical test behind them in this project (there is no ACF/PACF analysis in 04_EDA.ipynb).
5. **No centred rolling windows**, no backward filling, and no imputation using future data.
6. **Target and predictor construction are kept in separate functions**, so leakage between them can be checked directly.

## Target Variable

The dissertation's aim and research questions (RQ1–RQ3) specify forecasting Bitcoin and Ethereum **prices**. The correlation and Granger causality analysis (Section 4.7), however, established log returns — `np.log(price).diff()` — as the stationary representation of price, and every stationarity, correlation, and causality test in this dissertation was run on that transformation, not on raw price levels.

This notebook resolves that by treating log return as the **modelling target** (the quantity the statistical/ML/DL models are trained to predict, since it is the only stationary, evidence-grounded representation available) while separately carrying a raw price **reference column** (`{price}_level`, undifferenced, unlagged) through every feature set. That reference column is not a predictor and must be excluded from any model's input matrix; its only purpose is to let the evaluation stage reconstruct a price-level forecast from a predicted log return:

`predicted_price_t+h = price_level_t * exp(predicted_log_return_t+h)`

This way, RQ1–RQ3's price-forecasting aim can still be evaluated in price terms (e.g. price RMSE/MAPE) in Chapter 6, without abandoning the stationary log-return representation that the rest of the dissertation's statistical methodology depends on.

**The target is defined at three forecast horizons -- 1, 7, and 14 days -- not one.** For a given horizon `h`, the target is the *cumulative* log return over the next `h` days:

`target_t+h = log(price_t+h / price_t)`

This is not the same quantity as `log_return.shift(-h)` (a single day's return that happens to fall `h` days ahead) -- log returns are additive, so the cumulative return over `h` days equals the sum of the `h` intervening daily returns, and is computed directly here as a price-ratio log rather than summed lag-by-lag, which is simpler and numerically identical. At `h=1` this reduces exactly to the original one-day target (`log_return.shift(-1)`); at `h=7` and `h=14` it is the genuinely cumulative multi-day quantity. See `align_forecast_target()` in `src/analysis/feature_engineering.py` for the exact implementation, and the notebook title cell above for the full rationale behind this design.

## Building the Feature Sets

Four feature configurations are built per asset, not three. The fourth (Market + Google Trends + Fear & Greed Index) was added after review: RQ2 asks whether "incorporating selected external signals" -- both Google Trends and the Fear & Greed Index together -- improves forecasting compared with market data alone. The Market + Google Trends and Market + Fear & Greed Index configurations only test each signal on its own, so on their own they cannot answer RQ2 exactly as worded. The combined configuration fixes this by adding both signals to the Market Only baseline at the same time.

In [6]:
# Build all four feature configurations for each asset, at each of three
# forecast horizons (1, 7, 14 days) -- per the supervisor's request to
# compare across horizons, not just switch to a single new one. Internal
# keys (primary/candidate/ablation/combined) are retained because they
# match the CSV filenames produced below and the naming used by
# src/analysis/feature_engineering.py -- DISPLAY_NAMES maps each key to
# the descriptive label used in every printed output and saved table.
#
#   primary   = Market Only                                 = market data only (price + volume)
#   candidate = Market + Google Trends                      = primary + Google Trends
#   ablation  = Market + Fear & Greed Index                 = primary + Fear & Greed Index
#   combined  = Market + Google Trends + Fear & Greed Index = primary + both signals together
#               (this is the configuration that answers RQ2 exactly as worded)

DISPLAY_NAMES = {
    "primary": "Market Only",
    "candidate": "Market + Google Trends",
    "ablation": "Market + Fear & Greed Index",
    "combined": "Market + Google Trends + Fear & Greed Index",
}

# Forecast horizons in days, compared side by side per the supervisor's
# request. horizon=1 is the original design; 7 and 14 test whether the
# same conclusions (does tuning help, do external signals help, which
# model wins) hold as the forecasting horizon lengthens. Predictor
# construction is unaffected by horizon -- every predictor block is
# backward-looking only; only the target (and the persistence-baseline
# reference feature) depends on it.
FORECAST_HORIZONS = [1, 7, 14]

def horizon_suffix_for(horizon):
    """1-day keeps the original, unsuffixed filenames (nothing to protect
    against, since these ARE the original files); 7 and 14 get their own
    suffix so all three can coexist on disk without overwriting each other."""
    return f"_{horizon}day" if horizon != 1 else ""

all_feature_sets = {}  # keyed by (horizon, asset, feature_set_key)

for horizon in FORECAST_HORIZONS:
    for asset in ["Bitcoin", "Ethereum"]:

        primary_set = fe.build_feature_set(
            df, asset=asset, feature_set_name="primary",
            include_external=False, include_ablation=False, horizon=horizon,
        )
        all_feature_sets[(horizon, asset, "primary")] = primary_set

        candidate_set = fe.build_feature_set(
            df, asset=asset, feature_set_name="candidate",
            include_external=True, include_ablation=False, horizon=horizon,
        )
        all_feature_sets[(horizon, asset, "candidate")] = candidate_set

        ablation_set = fe.build_feature_set(
            df, asset=asset, feature_set_name="ablation",
            include_external=False, include_ablation=True, horizon=horizon,
        )
        all_feature_sets[(horizon, asset, "ablation")] = ablation_set

        combined_set = fe.build_feature_set(
            df, asset=asset, feature_set_name="combined",
            include_external=True, include_ablation=True, horizon=horizon,
        )
        all_feature_sets[(horizon, asset, "combined")] = combined_set

# Print the shape of every feature configuration at every horizon so we can see nothing went wrong.
for (horizon, asset, set_name), feature_set in all_feature_sets.items():
    n_columns = feature_set.shape[1] - 1  # minus the Date column
    print(f"h={horizon:2d}d | {asset:9s} | {DISPLAY_NAMES[set_name]:40s} | shape={feature_set.shape} | columns (excl. Date)={n_columns}")

h= 1d | Bitcoin   | Market Only                              | shape=(1799, 17) | columns (excl. Date)=16
h= 1d | Bitcoin   | Market + Google Trends                   | shape=(1799, 21) | columns (excl. Date)=20
h= 1d | Bitcoin   | Market + Fear & Greed Index              | shape=(1799, 23) | columns (excl. Date)=22
h= 1d | Bitcoin   | Market + Google Trends + Fear & Greed Index | shape=(1799, 27) | columns (excl. Date)=26
h= 1d | Ethereum  | Market Only                              | shape=(1799, 17) | columns (excl. Date)=16
h= 1d | Ethereum  | Market + Google Trends                   | shape=(1799, 21) | columns (excl. Date)=20
h= 1d | Ethereum  | Market + Fear & Greed Index              | shape=(1799, 23) | columns (excl. Date)=22
h= 1d | Ethereum  | Market + Google Trends + Fear & Greed Index | shape=(1799, 27) | columns (excl. Date)=26
h= 7d | Bitcoin   | Market Only                              | shape=(1799, 18) | columns (excl. Date)=17
h= 7d | Bitcoin   | Market + Google Tren

### Cross-asset features

As established in the feature-selection decision table above, no cross-asset lagged feature is created.

In [7]:
print(
    "Cross-asset lagged features were not created because cross-asset Granger "
    "causality was not established in the current implementation."
)


Cross-asset lagged features were not created because cross-asset Granger causality was not established in the current implementation.


## Leakage Validation

Each assembled feature set is checked against the standard set of leakage tests: chronological order, no duplicate dates, a spot-check that the evidence-selected lag feature exactly equals the source series shifted by that lag, that the target is correctly aligned to *t+h* for its own horizon (the cumulative *h*-day return, not a same-horizon single-day return), and that no infinite values are present. These checks run once per horizon, since the target itself differs by horizon even though the predictors do not.

This runs **before** any row removal below, while every feature set is still positionally aligned, row for row, with the source dataframe `df` — the checks compare each feature set against `df[column].shift(lag)`, so that alignment has to hold for the comparison to be meaningful.

In [8]:
leakage_results = []

for (horizon, asset, set_name), feature_set in all_feature_sets.items():
    checks = fe.run_leakage_checks(df, feature_set, asset=asset, horizon=horizon)
    checks["horizon"] = horizon
    checks["asset"] = asset
    checks["feature_set"] = DISPLAY_NAMES[set_name]
    leakage_results.append(checks)

leakage_results = pd.concat(leakage_results, ignore_index=True)
leakage_results = leakage_results[["horizon", "asset", "feature_set", "check", "passed"]]

display(leakage_results)

all_passed = bool(leakage_results["passed"].all())
print(f"\nAll leakage checks passed, across all {len(FORECAST_HORIZONS)} horizons: {all_passed}")

if not all_passed:
    raise AssertionError("One or more leakage checks failed. Stopping before saving any output.")

,horizon,asset,feature_set,check,passed
0,1,Bitcoin,Market Only,Chronological order preserved,True
1,1,Bitcoin,Market Only,No duplicate dates,True
2,1,Bitcoin,Market Only,Bitcoin_Price_log_return_lag_2 equals source s...,True
3,1,Bitcoin,Market Only,Target aligned to t+1 (cumulative 1-day return),True
4,1,Bitcoin,Market Only,No infinite values in feature set,True
...,...,...,...,...,...
131,14,Ethereum,Market + Google Trends + Fear & Greed Index,No duplicate dates,True
132,14,Ethereum,Market + Google Trends + Fear & Greed Index,Ethereum_Price_log_return_lag_2 equals source ...,True
133,14,Ethereum,Market + Google Trends + Fear & Greed Index,Target aligned to t+14 (cumulative 14-day return),True
134,14,Ethereum,Market + Google Trends + Fear & Greed Index,Ethereum_Price_trailing_14day_return uses only...,True



All leakage checks passed, across all 3 horizons: True


## Missing Values

Feature engineering introduces missing values by construction — the longest lag or rolling window used determines how many initial rows cannot be computed, and the target's final row is undefined (no *t+1* observation exists for the last date in the dataset). These structural gaps are unavoidable and expected.

The nine known missing volume observations (1–9 April 2026) no longer contribute additional gaps beyond this: they are linearly interpolated before any volume-derived feature is computed (see the Interpretation cell above), so every volume feature is fully populated except where structural lag/rolling lead-in already applies. The counts below therefore reflect structural missingness only, reported per feature set before any rows are removed, so the source of every missing value stays traceable rather than being cleared with a blanket `dropna()`.

In [9]:
missing_value_summary = []

for (horizon, asset, set_name), feature_set in all_feature_sets.items():
    missing_counts = feature_set.isnull().sum()
    missing_counts = missing_counts[missing_counts > 0]
    missing_value_summary.append({
        "horizon": horizon,
        "asset": asset,
        "feature_set": DISPLAY_NAMES[set_name],
        "n_columns_with_missing": len(missing_counts),
        "max_missing_in_a_column": int(missing_counts.max()) if len(missing_counts) else 0,
        "rows_with_any_missing": int(feature_set.isnull().any(axis=1).sum()),
    })

missing_value_summary = pd.DataFrame(missing_value_summary)
display(missing_value_summary)

,horizon,asset,feature_set,n_columns_with_missing,max_missing_in_a_column,rows_with_any_missing
0,1,Bitcoin,Market Only,14,31,32
1,1,Bitcoin,Market + Google Trends,18,31,32
2,1,Bitcoin,Market + Fear & Greed Index,20,31,32
3,1,Bitcoin,Market + Google Trends + Fear & Greed Index,24,31,32
4,1,Ethereum,Market Only,14,31,32
5,1,Ethereum,Market + Google Trends,18,31,32
6,1,Ethereum,Market + Fear & Greed Index,20,31,32
7,1,Ethereum,Market + Google Trends + Fear & Greed Index,24,31,32
8,7,Bitcoin,Market Only,15,31,38
9,7,Bitcoin,Market + Google Trends,19,31,38


#### Interpretation

With the nine-day volume gap now interpolated upstream (see above), the missing values remaining here have a single cause:

- **Structural NaNs** — the first *k* rows of any lag-*k* feature and the first *w* rows of any rolling-window-*w* feature, plus the final row of the target column (there is no *t+1* observation for the last date in the dataset). These are unavoidable and expected.

An earlier version of this notebook also reported a second category, **data-quality NaNs**, propagated from the nine known missing volume observations and able to affect up to 30 additional rows through the rolling window. That category no longer applies: interpolating the gap before any volume feature is computed means it never re-enters the pipeline as a missing value at all.

Only rows that remain unusable after **all** features and the target have been aligned are removed — not proactively. That row count is reported next, per feature set.

In [10]:
row_removal_summary = []

for (horizon, asset, set_name), feature_set in all_feature_sets.items():
    before = len(feature_set)
    cleaned = feature_set.dropna().reset_index(drop=True)
    after = len(cleaned)

    row_removal_summary.append({
        "horizon": horizon,
        "asset": asset,
        "feature_set": DISPLAY_NAMES[set_name],
        "rows_before": before,
        "rows_after": after,
        "rows_removed": before - after,
        "final_date_range": (str(cleaned["Date"].min().date()) + " to " + str(cleaned["Date"].max().date())) if after else "N/A",
    })

    all_feature_sets[(horizon, asset, set_name)] = cleaned

row_removal_summary = pd.DataFrame(row_removal_summary)
display(row_removal_summary)

,horizon,asset,feature_set,rows_before,rows_after,rows_removed,final_date_range
0,1,Bitcoin,Market Only,1799,1767,32,2021-07-02 to 2026-05-05
1,1,Bitcoin,Market + Google Trends,1799,1767,32,2021-07-02 to 2026-05-05
2,1,Bitcoin,Market + Fear & Greed Index,1799,1767,32,2021-07-02 to 2026-05-05
3,1,Bitcoin,Market + Google Trends + Fear & Greed Index,1799,1767,32,2021-07-02 to 2026-05-05
4,1,Ethereum,Market Only,1799,1767,32,2021-07-02 to 2026-05-05
5,1,Ethereum,Market + Google Trends,1799,1767,32,2021-07-02 to 2026-05-05
6,1,Ethereum,Market + Fear & Greed Index,1799,1767,32,2021-07-02 to 2026-05-05
7,1,Ethereum,Market + Google Trends + Fear & Greed Index,1799,1767,32,2021-07-02 to 2026-05-05
8,7,Bitcoin,Market Only,1799,1761,38,2021-07-02 to 2026-04-29
9,7,Bitcoin,Market + Google Trends,1799,1761,38,2021-07-02 to 2026-04-29


## Engineered Feature Dictionary

The following table documents every output column created by the feature-engineering process. For each feature, it records the source variable, transformation, lag or rolling window, feature-set membership, and methodological rationale.

This dictionary improves transparency and reproducibility by separating evidence-selected features from conventional forecasting features and experimental external variables. Target and raw price-reference columns are also identified explicitly so that they cannot be mistaken for model predictors.

In [11]:
def describe_feature(name: str, asset: str, feature_set: str) -> dict:
    """Infer a metadata row for one engineered feature from its naming convention."""

    if "target_t_plus" in name:
        # Extract the actual horizon from the column name itself (e.g. "t_plus_7"),
        # rather than assuming t+1 -- this notebook now builds targets at
        # multiple horizons, and a hardcoded "t+1" label would be wrong for
        # every column except the 1-day one.
        horizon_str = name.split("_target_t_plus_")[-1]
        transformation, lag_or_window, rationale = "cumulative log return, forward-shifted", f"t+{horizon_str}", "Forecasting target"
        source_variable = name.split("_log_return_target")[0]
    elif "trailing_" in name and "day_return" in name:
        transformation, source_variable = "cumulative trailing return", name.split("_trailing_")[0]
        lag_or_window = name.split("_trailing_")[-1].replace("day_return", "d")
        rationale = (
            "Added specifically for the Naive baseline at horizon>1 -- the "
            "direct analogue of 'yesterday predicts tomorrow' at a longer "
            "horizon is 'the last N days predict the next N days'. Also "
            "available to Random Forest, XGBoost, and LSTM like any other "
            "backward-looking feature, since there is no leakage reason to "
            "withhold it from them."
        )
    elif name.endswith("_level"):
        transformation, lag_or_window = "raw price level, unlagged", "N/A"
        source_variable = name.replace("_level", "")
        rationale = (
            "Reference column only — NOT a predictor. Carried forward so predicted log "
            "returns can be converted back into price-level forecasts during evaluation "
            "(the dissertation's aim is price forecasting; the models themselves are "
            "trained on log returns). Must be excluded from X in the modelling notebook."
        )
    elif "log_return" in name and "lag" in name:
        transformation, source_variable = "log return, lagged", name.split("_log_return_lag_")[0]
        lag_or_window = name.split("_lag_")[-1]
        rationale = "Evidence-selected (VAR/BIC)" if lag_or_window == str(config.EVIDENCE_SELECTED_LAG) else "Conventional lag, not evidence-based"
    elif "log_return" in name and "rolling" in name:
        transformation, source_variable = "log return, rolling statistic", name.split("_log_return_rolling")[0]
        lag_or_window = name.split("_")[-1]
        rationale = "EDA precedent — direct (30-day rolling window used for price/returns, Section 4.5.6)"
    elif "momentum" in name:
        transformation, source_variable = "price difference", name.split("_momentum_")[0]
        lag_or_window = name.split("_momentum_")[-1]
        rationale = "Conventional short-horizon comparison — arbitrary, not evidence-based"
    elif "Volume" in name and "rolling" in name:
        transformation, source_variable = "rolling statistic", name.split("_rolling")[0]
        lag_or_window = name.split("_")[-1]
        rationale = "EDA precedent — by analogy only (30-day window was used for price in Section 4.5.6, not volume specifically)"
    elif "Volume" in name:
        transformation = "raw lag" if "raw_lag" in name else (
            "pct change, lagged" if "pct_change" in name else "log level"
        )
        source_variable = name.split("_lag_")[0].split("_pct_change")[0].split("_log")[0]
        lag_or_window = name.split("_")[-1]
        rationale = "Stationarity analysis (first-differenced volume confirmed stationary) — not correlation/Granger evidence"
    elif "Search_Interest" in name and "diff" in name:
        transformation, source_variable = "diff (first difference), lagged", name.split("_diff_lag_")[0]
        lag_or_window = name.split("_diff_lag_")[-1]
        rationale = (
            "Evidence-selected (VAR/BIC), on the correctly differenced series — this is the "
            "feature that reproduces the exact variable-and-lag combination actually "
            "Granger-tested in 05_Correlation_Causality_analysis.ipynb"
            if lag_or_window == str(config.EVIDENCE_SELECTED_LAG)
            else "Conventional lag, differenced series, not the tested lag"
        )
    elif "Search_Interest" in name:
        transformation, source_variable = "raw level, lagged", name.split("_lag_")[0]
        lag_or_window = name.split("_lag_")[-1]
        rationale = (
            "NOT evidence-based — the Granger test used the differenced series, not this raw "
            "level, at any lag. Included for comparison only."
        )
    elif ("Fear_Greed" in name or "Market_Sentiment" in name) and "diff" in name:
        transformation, source_variable = "diff (first difference), lagged", name.split("_diff_lag_")[0]
        lag_or_window = name.split("_diff_lag_")[-1]
        rationale = (
            "Exploratory feature (Market + Fear & Greed Index configuration), evidence-selected lag on the differenced series — "
            "still not supported by the primary statistical screening (no significant "
            "correlation or Granger result for this variable at any lag)"
            if lag_or_window == str(config.EVIDENCE_SELECTED_LAG)
            else "Exploratory feature (Market + Fear & Greed Index configuration) — not supported by primary statistical screening"
        )
    elif "Fear_Greed" in name or "Market_Sentiment" in name:
        transformation = "rolling mean" if "rolling" in name else "raw level, lagged"
        source_variable = name.split("_lag_")[0].split("_rolling")[0]
        lag_or_window = name.split("_")[-1]
        rationale = "Exploratory feature (Market + Fear & Greed Index configuration) — not supported by primary statistical screening"
    else:
        transformation, source_variable, lag_or_window, rationale = "unknown", name, "N/A", "N/A"

    return {
        "feature_name": name,
        "asset": asset,
        "source_variable": source_variable,
        "transformation": transformation,
        "lag_or_window": lag_or_window,
        "feature_set": feature_set,
        "rationale": rationale,
    }


feature_metadata_by_horizon = {}

for horizon in FORECAST_HORIZONS:
    feature_metadata_rows = []
    for (h, asset, set_name), feature_set in all_feature_sets.items():
        if h != horizon:
            continue
        for column in feature_set.columns:
            if column == "Date":
                continue
            feature_metadata_rows.append(describe_feature(column, asset, DISPLAY_NAMES[set_name]))

    feature_metadata = pd.DataFrame(feature_metadata_rows).drop_duplicates(
        subset=["feature_name", "asset", "feature_set"]
    ).reset_index(drop=True)
    feature_metadata_by_horizon[horizon] = feature_metadata

    metadata_path = FEATURE_TABLES_DIR / f"feature_metadata{horizon_suffix_for(horizon)}.csv"
    feature_metadata.to_csv(metadata_path, index=False)
    print(f"Saved: {metadata_path}")

# horizon=1's table is used by the QC summary cell below, matching the
# original single-horizon variable name so that cell needs no further change.
feature_metadata = feature_metadata_by_horizon[1]
display(feature_metadata_by_horizon[FORECAST_HORIZONS[-1]])  # show the longest horizon's table as an example


Saved: /Users/mas364/Desktop/AI_Crypto_Forecasting/results/feature_engineering/tables/feature_metadata.csv
Saved: /Users/mas364/Desktop/AI_Crypto_Forecasting/results/feature_engineering/tables/feature_metadata_7day.csv
Saved: /Users/mas364/Desktop/AI_Crypto_Forecasting/results/feature_engineering/tables/feature_metadata_14day.csv


,feature_name,asset,source_variable,transformation,lag_or_window,feature_set,rationale
0,Bitcoin_Price_log_return,Bitcoin,Bitcoin_Price_log_return,unknown,N/A,Market Only,N/A
1,Bitcoin_Price_log_return_target_t_plus_14,Bitcoin,Bitcoin_Price,"cumulative log return, forward-shifted",t+14,Market Only,Forecasting target
2,Bitcoin_Price_level,Bitcoin,Bitcoin_Price,"raw price level, unlagged",N/A,Market Only,Reference column only — NOT a predictor. Carri...
3,Bitcoin_Price_log_return_lag_1,Bitcoin,Bitcoin_Price,"log return, lagged",1,Market Only,"Conventional lag, not evidence-based"
4,Bitcoin_Price_log_return_lag_2,Bitcoin,Bitcoin_Price,"log return, lagged",2,Market Only,Evidence-selected (VAR/BIC)
...,...,...,...,...,...,...,...
171,Fear_Greed_Index_lag_2,Ethereum,Fear_Greed_Index,"raw level, lagged",2,Market + Google Trends + Fear & Greed Index,Exploratory feature (Market + Fear & Greed Ind...
172,Fear_Greed_Index_diff_lag_1,Ethereum,Fear_Greed_Index,"diff (first difference), lagged",1,Market + Google Trends + Fear & Greed Index,Exploratory feature (Market + Fear & Greed Ind...
173,Fear_Greed_Index_diff_lag_2,Ethereum,Fear_Greed_Index,"diff (first difference), lagged",2,Market + Google Trends + Fear & Greed Index,Exploratory feature (Market + Fear & Greed Ind...
174,Fear_Greed_Index_rolling_mean_7,Ethereum,Fear_Greed_Index,rolling mean,7,Market + Google Trends + Fear & Greed Index,Exploratory feature (Market + Fear & Greed Ind...


In [12]:
# Display the complete engineered-feature dictionary.
feature_dictionary_columns = [
    "feature_name",
    "asset",
    "source_variable",
    "transformation",
    "lag_or_window",
    "feature_set",
    "rationale",
]

display(
    feature_metadata[feature_dictionary_columns]
    .sort_values(["asset", "feature_set", "feature_name"])
    .reset_index(drop=True)
)

,feature_name,asset,source_variable,transformation,lag_or_window,feature_set,rationale
0,Bitcoin_Price_level,Bitcoin,Bitcoin_Price,"raw price level, unlagged",N/A,Market + Fear & Greed Index,Reference column only — NOT a predictor. Carri...
1,Bitcoin_Price_log_return,Bitcoin,Bitcoin_Price_log_return,unknown,N/A,Market + Fear & Greed Index,N/A
2,Bitcoin_Price_log_return_lag_1,Bitcoin,Bitcoin_Price,"log return, lagged",1,Market + Fear & Greed Index,"Conventional lag, not evidence-based"
3,Bitcoin_Price_log_return_lag_2,Bitcoin,Bitcoin_Price,"log return, lagged",2,Market + Fear & Greed Index,Evidence-selected (VAR/BIC)
4,Bitcoin_Price_log_return_lag_7,Bitcoin,Bitcoin_Price,"log return, lagged",7,Market + Fear & Greed Index,"Conventional lag, not evidence-based"
...,...,...,...,...,...,...,...
163,Ethereum_Volume_pct_change_lag_7,Ethereum,Ethereum_Volume,"pct change, lagged",7,Market Only,Stationarity analysis (first-differenced volum...
164,Ethereum_Volume_raw_lag_1,Ethereum,Ethereum_Volume_raw,raw lag,1,Market Only,Stationarity analysis (first-differenced volum...
165,Ethereum_Volume_raw_lag_7,Ethereum,Ethereum_Volume_raw,raw lag,7,Market Only,Stationarity analysis (first-differenced volum...
166,Ethereum_Volume_rolling_mean_30,Ethereum,Ethereum_Volume,rolling statistic,30,Market Only,EDA precedent — by analogy only (30-day window...


## Saving the Model-Ready Datasets

In [13]:
# Filenames carry a horizon suffix (blank for 1-day, "_7day", "_14day")
# so all three horizons worth of feature files coexist on disk without
# overwriting one another. horizon=1 uses the original, unsuffixed
# filenames -- these ARE the project's original files, not a new copy of them.

for horizon in FORECAST_HORIZONS:
    suffix = horizon_suffix_for(horizon)
    output_filenames = {
        ("Bitcoin", "primary"): f"bitcoin_market_only_features{suffix}.csv",
        ("Bitcoin", "candidate"): f"bitcoin_market_google_trends_features{suffix}.csv",
        ("Bitcoin", "ablation"): f"bitcoin_market_fear_greed_features{suffix}.csv",
        ("Bitcoin", "combined"): f"bitcoin_market_all_external_features{suffix}.csv",
        ("Ethereum", "primary"): f"ethereum_market_only_features{suffix}.csv",
        ("Ethereum", "candidate"): f"ethereum_market_google_trends_features{suffix}.csv",
        ("Ethereum", "ablation"): f"ethereum_market_fear_greed_features{suffix}.csv",
        ("Ethereum", "combined"): f"ethereum_market_all_external_features{suffix}.csv",
    }

    for (asset, set_name), filename in output_filenames.items():
        output_path = FEATURES_DIR / filename
        all_feature_sets[(horizon, asset, set_name)].to_csv(output_path, index=False)
        print(f"Saved: {output_path}  (shape={all_feature_sets[(horizon, asset, set_name)].shape})")

Saved: /Users/mas364/Desktop/AI_Crypto_Forecasting/data/processed/features/bitcoin_market_only_features.csv  (shape=(1767, 17))
Saved: /Users/mas364/Desktop/AI_Crypto_Forecasting/data/processed/features/bitcoin_market_google_trends_features.csv  (shape=(1767, 21))
Saved: /Users/mas364/Desktop/AI_Crypto_Forecasting/data/processed/features/bitcoin_market_fear_greed_features.csv  (shape=(1767, 23))
Saved: /Users/mas364/Desktop/AI_Crypto_Forecasting/data/processed/features/bitcoin_market_all_external_features.csv  (shape=(1767, 27))
Saved: /Users/mas364/Desktop/AI_Crypto_Forecasting/data/processed/features/ethereum_market_only_features.csv  (shape=(1767, 17))
Saved: /Users/mas364/Desktop/AI_Crypto_Forecasting/data/processed/features/ethereum_market_google_trends_features.csv  (shape=(1767, 21))
Saved: /Users/mas364/Desktop/AI_Crypto_Forecasting/data/processed/features/ethereum_market_fear_greed_features.csv  (shape=(1767, 23))
Saved: /Users/mas364/Desktop/AI_Crypto_Forecasting/data/process

## Correlation Among Engineered Predictors

High correlation between engineered features (for example, `_lag_1` and `_lag_2` of the same source series) is expected and is not automatically removed here — different models tolerate collinearity differently, and removing a feature at this stage would pre-judge a decision that belongs in model development. Pairs above 0.8 are flagged for review instead.

In [14]:
def flag_collinear_pairs(feature_set: pd.DataFrame, threshold: float = 0.8) -> pd.DataFrame:
    """Return predictor pairs whose absolute Pearson correlation exceeds `threshold`."""

    predictors = feature_set.drop(columns=["Date"]).select_dtypes(include=[np.number])
    predictors = predictors.drop(
        columns=[c for c in predictors.columns if "target_t_plus" in c], errors="ignore"
    )
    corr_matrix = predictors.corr()

    pairs = []
    columns = corr_matrix.columns
    for i, col_a in enumerate(columns):
        for col_b in columns[i + 1:]:
            r = corr_matrix.loc[col_a, col_b]
            if pd.notna(r) and abs(r) >= threshold:
                pairs.append({"Feature A": col_a, "Feature B": col_b, "Correlation": round(r, 3)})

    return pd.DataFrame(pairs).sort_values("Correlation", key=abs, ascending=False) if pairs else pd.DataFrame(
        columns=["Feature A", "Feature B", "Correlation"]
    )


# Checked once, at horizon=1 only -- every predictor is backward-looking and
# identical in construction regardless of horizon (only the target column
# differs between horizons, and it is already excluded from this check), so
# collinearity among predictors would be identical at every horizon. Checking
# all three would just repeat the same result three times.
for (horizon, asset, set_name), feature_set in all_feature_sets.items():
    if horizon != 1:
        continue
    collinear = flag_collinear_pairs(feature_set)
    print(f"\n{asset} — {set_name} ({len(collinear)} pairs with |r| >= 0.8):")
    if len(collinear):
        display(collinear)



Bitcoin — primary (0 pairs with |r| >= 0.8):

Bitcoin — candidate (0 pairs with |r| >= 0.8):

Bitcoin — ablation (6 pairs with |r| >= 0.8):


,Feature A,Feature B,Correlation
3,Fear_Greed_Index_lag_2,Fear_Greed_Index_rolling_mean_7,0.969
2,Fear_Greed_Index_lag_1,Market_Sentiment_Encoded_lag_1,0.966
0,Fear_Greed_Index_lag_1,Fear_Greed_Index_lag_2,0.963
1,Fear_Greed_Index_lag_1,Fear_Greed_Index_rolling_mean_7,0.955
4,Fear_Greed_Index_lag_2,Market_Sentiment_Encoded_lag_1,0.931
5,Fear_Greed_Index_rolling_mean_7,Market_Sentiment_Encoded_lag_1,0.923



Bitcoin — combined (6 pairs with |r| >= 0.8):


,Feature A,Feature B,Correlation
3,Fear_Greed_Index_lag_2,Fear_Greed_Index_rolling_mean_7,0.969
2,Fear_Greed_Index_lag_1,Market_Sentiment_Encoded_lag_1,0.966
0,Fear_Greed_Index_lag_1,Fear_Greed_Index_lag_2,0.963
1,Fear_Greed_Index_lag_1,Fear_Greed_Index_rolling_mean_7,0.955
4,Fear_Greed_Index_lag_2,Market_Sentiment_Encoded_lag_1,0.931
5,Fear_Greed_Index_rolling_mean_7,Market_Sentiment_Encoded_lag_1,0.923



Ethereum — primary (0 pairs with |r| >= 0.8):

Ethereum — candidate (1 pairs with |r| >= 0.8):


,Feature A,Feature B,Correlation
0,Ethereum_Search_Interest_lag_1,Ethereum_Search_Interest_lag_2,0.896



Ethereum — ablation (6 pairs with |r| >= 0.8):


,Feature A,Feature B,Correlation
3,Fear_Greed_Index_lag_2,Fear_Greed_Index_rolling_mean_7,0.969
2,Fear_Greed_Index_lag_1,Market_Sentiment_Encoded_lag_1,0.966
0,Fear_Greed_Index_lag_1,Fear_Greed_Index_lag_2,0.963
1,Fear_Greed_Index_lag_1,Fear_Greed_Index_rolling_mean_7,0.955
4,Fear_Greed_Index_lag_2,Market_Sentiment_Encoded_lag_1,0.931
5,Fear_Greed_Index_rolling_mean_7,Market_Sentiment_Encoded_lag_1,0.923



Ethereum — combined (7 pairs with |r| >= 0.8):


,Feature A,Feature B,Correlation
4,Fear_Greed_Index_lag_2,Fear_Greed_Index_rolling_mean_7,0.969
3,Fear_Greed_Index_lag_1,Market_Sentiment_Encoded_lag_1,0.966
1,Fear_Greed_Index_lag_1,Fear_Greed_Index_lag_2,0.963
2,Fear_Greed_Index_lag_1,Fear_Greed_Index_rolling_mean_7,0.955
5,Fear_Greed_Index_lag_2,Market_Sentiment_Encoded_lag_1,0.931
6,Fear_Greed_Index_rolling_mean_7,Market_Sentiment_Encoded_lag_1,0.923
0,Ethereum_Search_Interest_lag_1,Ethereum_Search_Interest_lag_2,0.896


#### Interpretation

Highly correlated engineered features here are expected and mechanical (a variable's own lag-1 and lag-2 values, or its rolling mean and rolling std, are inherently related) rather than a sign of a problem. Whether removal is necessary depends on the model:

- **ARIMA** is univariate by construction; only the target's own history is used, so collinearity between engineered predictor columns is not directly relevant to it. If any external or volume variable is added, this becomes ARIMAX rather than ARIMA, and multicollinearity between exogenous regressors would then matter in the usual OLS sense.
- **Random Forest and XGBoost** are largely robust to collinear predictors — tree splits are not destabilised by correlated features the way linear coefficients are — so removal is not generally required, though redundant features add training time for no benefit.
- **LSTM and Transformer** models can absorb correlated inputs without a strict collinearity requirement, but redundant channels increase input dimensionality unnecessarily; a lighter feature set is often preferable for these architectures specifically for efficiency, not correctness.

No feature is removed in this notebook on the basis of collinearity; this is left as a modelling-stage decision, made per model rather than once globally.

## Feature Engineering Limitations

The feature-engineering strategy is informed mainly by stationarity, correlation, and Granger causality analyses. Correlation and standard Granger tests assess linear or monotonic relationships and may not detect nonlinear, regime-dependent, or interaction effects. The absence of statistical significance should therefore not be interpreted as proof that an external variable has no forecasting value.

The selected lag of two days follows the BIC result from the earlier VAR analysis. Lag 1 and lag 7 are retained as conventional comparison lags rather than evidence-selected lags, and their contribution must be assessed using validation data during modelling.

Rolling features summarise recent behaviour but require the loss of early observations and may smooth short-lived market changes. Their usefulness must therefore be established through out-of-sample performance rather than assumed from their construction.

The Google Trends series is independently normalised, meaning that Bitcoin and Ethereum search-interest levels should not be interpreted as directly comparable absolute quantities. The external-feature experiments should consequently be interpreted as within-series forecasting tests rather than direct measures of relative public interest.

Finally, feature construction is performed before model-specific selection. Redundant engineered features are flagged but not removed globally because different model classes respond differently to correlated inputs. Any further selection must be conducted using training and validation data only, without consulting the held-out test set.

## Model-Specific Feature Set Notes

The feature sets built in this notebook are tabular (one row per date). How each planned model consumes them differs, and constructing that final form belongs in `07_modeling.ipynb`, not here:

- **Naïve baseline** — uses only `{price}_log_return_lag_1` (or the raw price itself, shifted by one day); no other engineered feature is required.
- **ARIMA** — univariate on the target's own history (`{price}_log_return`); the lag structure is chosen internally by the ARIMA order search, not by the lag columns built here. If any external, volume, or sentiment feature is added, this becomes ARIMAX, and that distinction should be stated explicitly in the modelling chapter.
- **Random Forest / XGBoost** — can consume the Market Only, Market + Google Trends, Market + Fear & Greed Index, or Market + Google Trends + Fear & Greed Index configurations directly as tabular input, dropping the Date column and any rows containing the target's own current-period log return where required.
- **LSTM / Transformer** — need this same tabular data reshaped into overlapping sequence windows (samples × timesteps × features). That reshaping is a modelling-stage step and is intentionally not implemented in this notebook, per the current scope of feature engineering.

## Quality Control Summary

In [15]:
print("=" * 70)
print("FEATURE ENGINEERING — QUALITY CONTROL SUMMARY")
print("=" * 70)

print(f"\nOriginal dataset shape (post market-cap removal): {df.shape}")

print("\nFinal shape per feature configuration, per horizon:")
for (horizon, asset, set_name), feature_set in all_feature_sets.items():
    n_missing = int(feature_set.isnull().sum().sum())
    n_infinite = int(np.isinf(feature_set.select_dtypes(include=[np.number])).sum().sum())
    print(
        f"  h={horizon:2d}d | {asset:9s} | {DISPLAY_NAMES[set_name]:40s} | shape={feature_set.shape} | "
        f"missing={n_missing} | infinite={n_infinite} | "
        f"date range={feature_set['Date'].min().date()} to {feature_set['Date'].max().date()} | "
        f"duplicate dates={int(feature_set['Date'].duplicated().sum())}"
    )

print(f"\nForecast horizons compared: {FORECAST_HORIZONS} days")
print("Target definition per horizon: cumulative log return, "
      "target_t_plus_H = log(price.shift(-H) / price)")

print("\nFear_Greed_Index and Market_Sentiment_Encoded appear only in the Market + Fear & Greed Index")
print("  and Market + Google Trends + Fear & Greed Index configurations, never in Market Only or")
print("  Market + Google Trends; Bitcoin_Market_Cap and Ethereum_Market_Cap (removed entirely);")
print("  cross-asset lagged features (not created).")

print(f"\nAll leakage checks passed, across all horizons: {bool(leakage_results['passed'].all())}")

FEATURE ENGINEERING — QUALITY CONTROL SUMMARY

Original dataset shape (post market-cap removal): (1799, 10)

Final shape per feature configuration, per horizon:
  h= 1d | Bitcoin   | Market Only                              | shape=(1767, 17) | missing=0 | infinite=0 | date range=2021-07-02 to 2026-05-05 | duplicate dates=0
  h= 1d | Bitcoin   | Market + Google Trends                   | shape=(1767, 21) | missing=0 | infinite=0 | date range=2021-07-02 to 2026-05-05 | duplicate dates=0
  h= 1d | Bitcoin   | Market + Fear & Greed Index              | shape=(1767, 23) | missing=0 | infinite=0 | date range=2021-07-02 to 2026-05-05 | duplicate dates=0
  h= 1d | Bitcoin   | Market + Google Trends + Fear & Greed Index | shape=(1767, 27) | missing=0 | infinite=0 | date range=2021-07-02 to 2026-05-05 | duplicate dates=0
  h= 1d | Ethereum  | Market Only                              | shape=(1767, 17) | missing=0 | infinite=0 | date range=2021-07-02 to 2026-05-05 | duplicate dates=0
  h= 1d | E

## Feature-Set Manifest and Cross-Configuration Alignment Verification

Phase 2 audit requirement: before this notebook's outputs are used downstream, this section explicitly verifies that all four feature configurations for a given asset share identical dates and identical target values -- not just an assumption carried over from the leakage checks earlier in this notebook, but a direct row-by-row comparison. A manifest recording each configuration's shape, date range, target column, price-reference column, and predictor names is then saved for later reference by Notebooks 07 and 08.

In [16]:
alignment_checks = []

for horizon in FORECAST_HORIZONS:
    for asset in ["Bitcoin", "Ethereum"]:
        asset_sets = {name: fs for (h, a, name), fs in all_feature_sets.items() if a == asset and h == horizon}
        reference_name = "primary"
        reference_dates = asset_sets[reference_name]["Date"].reset_index(drop=True)

        for set_name, fs in asset_sets.items():
            dates_match = fs["Date"].reset_index(drop=True).equals(reference_dates)
            alignment_checks.append({
                "horizon": horizon, "asset": asset, "feature_set_key": set_name,
                "dates_identical_to_primary": dates_match,
                "row_count": len(fs),
            })

alignment_df = pd.DataFrame(alignment_checks)
display(alignment_df)

assert alignment_df["dates_identical_to_primary"].all(), "Date misalignment found across feature configurations -- stopping before saving manifest."
print("\nAll four feature configurations share identical dates within each asset, at every horizon.")

,horizon,asset,feature_set_key,dates_identical_to_primary,row_count
0,1,Bitcoin,primary,True,1767
1,1,Bitcoin,candidate,True,1767
2,1,Bitcoin,ablation,True,1767
3,1,Bitcoin,combined,True,1767
4,1,Ethereum,primary,True,1767
5,1,Ethereum,candidate,True,1767
6,1,Ethereum,ablation,True,1767
7,1,Ethereum,combined,True,1767
8,7,Bitcoin,primary,True,1761
9,7,Bitcoin,candidate,True,1761



All four feature configurations share identical dates within each asset, at every horizon.


In [17]:
# Verify target values are identical across feature sets for the same asset,
# at each horizon separately (the target is asset-and-horizon-specific, not
# feature-set-specific, so it must not vary across feature sets within a
# fixed asset and horizon).

target_alignment_checks = []

for horizon in FORECAST_HORIZONS:
    for asset in ["Bitcoin", "Ethereum"]:
        price_col = f"{asset}_Price"
        target_col = f"{price_col}_log_return_target_t_plus_{horizon}"

        asset_sets = {name: fs for (h, a, name), fs in all_feature_sets.items() if a == asset and h == horizon}
        reference_target = asset_sets["primary"][target_col].reset_index(drop=True)

        for set_name, fs in asset_sets.items():
            target_series = fs[target_col].reset_index(drop=True)
            identical = np.allclose(reference_target.values, target_series.values, equal_nan=True)
            target_alignment_checks.append({
                "horizon": horizon, "asset": asset, "feature_set_key": set_name,
                "target_column": target_col,
                "target_identical_to_primary": identical,
            })

target_alignment_df = pd.DataFrame(target_alignment_checks)
display(target_alignment_df)

assert target_alignment_df["target_identical_to_primary"].all(), "Target values differ across feature configurations -- stopping before saving manifest."
print("\nTarget values confirmed identical across all four feature configurations, per asset, at every horizon.")

,horizon,asset,feature_set_key,target_column,target_identical_to_primary
0,1,Bitcoin,primary,Bitcoin_Price_log_return_target_t_plus_1,True
1,1,Bitcoin,candidate,Bitcoin_Price_log_return_target_t_plus_1,True
2,1,Bitcoin,ablation,Bitcoin_Price_log_return_target_t_plus_1,True
3,1,Bitcoin,combined,Bitcoin_Price_log_return_target_t_plus_1,True
4,1,Ethereum,primary,Ethereum_Price_log_return_target_t_plus_1,True
5,1,Ethereum,candidate,Ethereum_Price_log_return_target_t_plus_1,True
6,1,Ethereum,ablation,Ethereum_Price_log_return_target_t_plus_1,True
7,1,Ethereum,combined,Ethereum_Price_log_return_target_t_plus_1,True
8,7,Bitcoin,primary,Bitcoin_Price_log_return_target_t_plus_7,True
9,7,Bitcoin,candidate,Bitcoin_Price_log_return_target_t_plus_7,True



Target values confirmed identical across all four feature configurations, per asset, at every horizon.


In [18]:
# Build and save the feature-set manifest -- one combined manifest covering
# all three horizons, with a "horizon" column, so a later notebook can filter
# or compare across horizons directly from this single file.

manifest_rows = []

for (horizon, asset, set_name), fs in all_feature_sets.items():
    price_col = f"{asset}_Price"
    target_col = f"{price_col}_log_return_target_t_plus_{horizon}"
    level_col = f"{price_col}_level"
    predictor_names = [c for c in fs.columns if c not in ("Date", level_col, target_col)]

    manifest_rows.append({
        "horizon": horizon,
        "asset": asset,
        "feature_set_key": set_name,
        "feature_set_name": DISPLAY_NAMES[set_name],
        "row_count": len(fs),
        "date_range_start": str(fs["Date"].min().date()),
        "date_range_end": str(fs["Date"].max().date()),
        "target_column": target_col,
        "price_reference_column": level_col,
        "predictor_count": len(predictor_names),
        "predictor_names": predictor_names,
    })

feature_set_manifest = pd.DataFrame(manifest_rows)

manifest_path = FEATURE_TABLES_DIR / "feature_set_manifest_all_horizons.json"
feature_set_manifest.to_json(manifest_path, orient="records", indent=2)
print(f"Saved: {manifest_path}")

display(feature_set_manifest[["horizon", "asset", "feature_set_key", "feature_set_name", "row_count", "date_range_start", "date_range_end", "predictor_count"]])

Saved: /Users/mas364/Desktop/AI_Crypto_Forecasting/results/feature_engineering/tables/feature_set_manifest_all_horizons.json


,horizon,asset,feature_set_key,feature_set_name,row_count,date_range_start,date_range_end,predictor_count
0,1,Bitcoin,primary,Market Only,1767,2021-07-02,2026-05-05,14
1,1,Bitcoin,candidate,Market + Google Trends,1767,2021-07-02,2026-05-05,18
2,1,Bitcoin,ablation,Market + Fear & Greed Index,1767,2021-07-02,2026-05-05,20
3,1,Bitcoin,combined,Market + Google Trends + Fear & Greed Index,1767,2021-07-02,2026-05-05,24
4,1,Ethereum,primary,Market Only,1767,2021-07-02,2026-05-05,14
5,1,Ethereum,candidate,Market + Google Trends,1767,2021-07-02,2026-05-05,18
6,1,Ethereum,ablation,Market + Fear & Greed Index,1767,2021-07-02,2026-05-05,20
7,1,Ethereum,combined,Market + Google Trends + Fear & Greed Index,1767,2021-07-02,2026-05-05,24
8,7,Bitcoin,primary,Market Only,1761,2021-07-02,2026-04-29,15
9,7,Bitcoin,candidate,Market + Google Trends,1761,2021-07-02,2026-04-29,19


## Methodological Summary

**Statement under review:**

> "Feature engineering was informed by a two-stage feature selection strategy. Statistical evidence from stationarity, correlation, and Granger causality analyses was first used to identify redundant, unsupported, and potentially informative variables. Candidate variables were then retained for controlled empirical comparison during modelling."

**Verdict: Partially supported.**

The first half of the statement is now fully supported by this notebook: the feature-selection decision table above is built directly from the stationarity, correlation, and Granger causality results, and every inclusion or exclusion decision — Fear & Greed excluded, Google Trends retained only as a candidate, volume retained on stationarity grounds alone, cross-asset features withheld — is traceable to a specific number in those tables, not to assumption.

The second half — "candidate variables were then retained for controlled empirical comparison during modelling" — is not yet supported, because the modelling stage (`07_modeling.ipynb`) has not been implemented. This notebook produces four datasets per asset (Market Only, Market + Google Trends, Market + Fear & Greed Index, and Market + Google Trends + Fear & Greed Index) so that modelling can compare each external signal on its own and both signals together against the market-data-only baseline — which is what RQ2 actually asks. Whether that comparison is carried out, and whether its result is reported honestly regardless of outcome, can only be confirmed once `07_modeling.ipynb` exists. The statement will become fully supported only when that empirical comparison is actually performed and its result — including a null result, if that is what occurs, consistent with the pattern already seen in the correlation and Granger analysis — is reported in Chapter 5/6.

## Final Assumptions Before Modelling

The following decisions were made in this notebook without explicit support elsewhere in the project and should be confirmed before modelling proceeds:

1. **Forecasting target (return vs. price level) — resolved.** The dissertation aim specifies price forecasting; this notebook now trains on log returns (the only stationary, evidence-grounded representation) while carrying a `{price}_level` reference column for converting predictions back to price terms in evaluation. Confirm this reconstruction approach is what `07_modeling.ipynb` and `08_evaluation.ipynb` should use, rather than training directly on price levels.
2. **Cross-asset Granger causality.** Bitcoin and Ethereum returns are strongly correlated (r = 0.838) but this relationship has never been tested for Granger causality in either direction. No cross-asset feature is created as a result; this is a gap worth closing in the causality analysis notebook before, not after, cross-asset features are considered.
3. **Ordinal sentiment encoding.** `Market_Sentiment_Encoded` treats the five Fear & Greed categories as equally spaced (0 to 4). This assumption is inherited from the preprocessing stage and is not tested anywhere in this project; it is used as-is in the Market + Fear & Greed Index configuration, not re-validated here.
4. **AIC-preferred lag (5) was not used.** The causality analysis notebook noted that AIC and FPE preferred a 5-day lag while BIC and HQIC preferred 2 days, and used BIC as the primary criterion. This notebook follows that same choice (lag 2) throughout. A 5-day lag was not tested as an alternative evidence-selected lag here; doing so would require re-running the Granger tests in the causality notebook first, not assuming a different lag in this one.